In [4]:
import pandas as pd
import fastembed 
import qdrant_client

# FASE 1 INDICIZZAZIONE

In [5]:
import qdrant_client
from qdrant_client import models
client = qdrant_client.QdrantClient('http://localhost:6333', timeout=1000)

In [6]:
from fastembed import TextEmbedding, SparseTextEmbedding, LateInteractionTextEmbedding

dense_embedding_model = TextEmbedding("jinaai/jina-embeddings-v3", cache_dir = './fastembed/')
bm25_embedding_model = SparseTextEmbedding("Qdrant/bm25", cache_dir = './fastembed/')
late_interaction_embedding_model = LateInteractionTextEmbedding("colbert-ir/colbertv2.0", cache_dir = './fastembed/')

C:\Users\CT-01\PycharmProjects\rag\venv\Lib\site-packages\qdrant_client\qdrant_remote.py:288: UserWarning: Failed to obtain server version. Unable to check client-server compatibility. Set check_compatibility=False to skip version check.
  show_warning(


In [37]:
client.create_collection(
            collection_name='rag_capitoli',
            vectors_config={
                "dense": models.VectorParams(
                    size=1024,
                    distance=models.Distance.COSINE
                ),
                 "colbert": models.VectorParams(
                size=128,
                distance=models.Distance.COSINE,
                multivector_config=models.MultiVectorConfig(
                    comparator=models.MultiVectorComparator.MAX_SIM
                ),
                hnsw_config=models.HnswConfigDiff(m=0)  # Disable HNSW for reranking
        )
                
            },
    sparse_vectors_config={
        "bm25": models.SparseVectorParams(modifier=models.Modifier.IDF)
    }
)

True

In [125]:
df = pd.read_excel('../data/INDICI_PULITI_MVP_01.xlsx', sheet_name=7)

In [126]:
df.head()

,id_indice,disciplina,ordine_scuola,liv_min,liv_max,unita,lezione,sotto_lezione,descrizione_breve,tag,is_uploaded
0,PRI-1-CIT-001,Cittadinanza e Costituzione,Scuola Primaria,0,0,Scopro e imparo le prime regole,Conoscere le prime formazioni sociali,NaN,Si esplorano le prime forme di aggregazione um...,"formazioni sociali, famiglia, gruppo, società,...",True
1,PRI-1-CIT-002,Cittadinanza e Costituzione,Scuola Primaria,0,0,Scopro e imparo le prime regole,Conoscere e rispettare le regole di convivenza,NaN,Si apprendono le norme fondamentali per vivere...,"regole, convivenza, rispetto, norme sociali, e...",True
2,PRI-1-CIT-003,Cittadinanza e Costituzione,Scuola Primaria,0,0,Scopro e imparo le prime regole,Conoscere e rispettare i simboli costituzional...,NaN,Si studiano i simboli che rappresentano l'Ital...,"simboli nazionali, Costituzione, Repubblica It...",True
3,PRI-1-CIT-004,Cittadinanza e Costituzione,Scuola Primaria,0,0,Scopro e imparo le prime regole,Conoscere e rispettare le norme per la tutela ...,NaN,Si imparano le regole per proteggere l'ambient...,"tutela paesaggio, patrimonio storico, ambiente...",True
4,PRI-1-CIT-005,Cittadinanza e Costituzione,Scuola Primaria,0,0,Scopro e imparo le prime regole,"Conoscere, accettare e rispettare le principal...",NaN,Si apprendono le regole fondamentali del codic...,"educazione stradale, sicurezza stradale, codic...",True


In [127]:
id_indices = list(df.id_indice)

In [128]:
len(id_indices)

322

In [130]:
disciplina = 'cittadinanza'
ordine_scuola = list(df.ordine_scuola)
liv_min = list(df.liv_min)
liv_max = list(df.liv_max)
unita = list(df.unita)
lezione = list(df.lezione)
titolo = [unit + '\n' + lez for unit, lez in zip(unita, lezione)]
sotto_lezione = [slez if type(slez) == str else '' for slez in df.sotto_lezione ]
descrizione_breve = list(df.descrizione_breve)
descrizione = [unit + '\n' + lez + '\n' + sotto_lez + '\n' + descr for unit, lez, sotto_lez, descr in zip(unita, lezione, sotto_lezione, descrizione_breve)]
tag = list(df.tag)

In [131]:
dense_embeddings = list(dense_embedding_model.embed(text for text in descrizione))

In [132]:
late_interaction_embeddings = list(late_interaction_embedding_model.embed(text for text in descrizione))

In [133]:
bm25_embeddings = list(bm25_embedding_model.embed(text for text in descrizione))

In [134]:
from qdrant_client.models import PointStruct

points = []
for indice,(idx, dense_embedding, bm25_embedding, late_interaction_embedding, tit, ord_scuola, liv_minimo, liv_maximo, descr, tg) in enumerate(zip(id_indices, dense_embeddings, bm25_embeddings, late_interaction_embeddings, titolo, ordine_scuola, liv_min, liv_max,  descrizione, tag)):
  
    point = PointStruct(
        id=indice+1220,
        vector={
            "dense": dense_embedding,
            "colbert": late_interaction_embedding,
            "bm25": bm25_embedding.as_object(),
        },
        payload={"indice":idx, "descr": descr, "disciplina": disciplina, "ordine_scuola": ord_scuola, "liv_min": liv_minimo, "liv_max": liv_maximo, "tag": tg, "titolo": tit}
    )
    points.append(point)

In [135]:
for batch in range(len(points) // 20):
    operation_info = client.upsert(
    collection_name="rag_capitoli",
    points=points[batch * 20:(batch+1) * 20]
)
operation_info = client.upsert(
    collection_name="rag_capitoli",
    points=points[batch * 20:])
print(operation_info)

operation_id=113 status=<UpdateStatus.COMPLETED: 'completed'>


# FASE 2: FASE QUERY

In [7]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    'jinaai/jina-reranker-v3',
    dtype="auto",
    trust_remote_code=True,
)
model.eval()

Loading weights:   0%|          | 0/312 [00:00<?, ?it/s]

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [8]:
import torch
torch.device('cuda' if torch.cuda.is_available() else 'cpu')


device(type='cuda')

In [9]:
model.to('cuda')

JinaForRanking(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layerno

In [9]:
#query = "Si descrive Platone e il mondo delle idee"
query = "Il prof. Matteo Saudino, ideatore del canale YouTube <i>BarbaSophia</i>, legge un passaggio del Manifesto ed Engels in cui si desrivono il ruolo e la trasformazione della borghesia."

In [10]:
dense_vector = next(dense_embedding_model.query_embed(query))
sparse_vector = next(bm25_embedding_model.query_embed(query))
late_interaction_vector = next(late_interaction_embedding_model.query_embed(query))

In [11]:
prefetch = [
        models.Prefetch(
            query=dense_vector,
            using="dense",
            limit=20,
        ),
        models.Prefetch(
            query=models.SparseVector(**sparse_vector.as_object()),
            using="bm25",
            limit=1,
        ),
    ]

In [12]:
results = client.query_points(
         "rag_capitoli",
        prefetch=prefetch,
        query=late_interaction_vector,
        using="colbert",
        with_payload=True,
        limit=20,
)

points = results.points

In [13]:
#model.to('cuda')
results = model.rerank(query, [point.payload['descr'] for point in points])

# Results are sorted by relevance score (highest first)
for result in results:
    print(f"Score: {result['relevance_score']:.4f}")
    print(f"Document: {result['document'][:100]}...")
    print()

Score: 0.1647
Document: Marx e il materialismo storico
IL SOCIALISMO SCIENTIFICO
Il sostegno economico di Engels
L'amicizia ...

Score: 0.1032
Document: Marx e il materialismo storico
Il Manifesto del partito comunista

Presenta il "Manifesto del partit...

Score: 0.0013
Document: Marx e il materialismo storico
Economia borghese e alienazione
I quattro aspetti dell’alienazione
De...

Score: -0.0083
Document: Marx e il materialismo storico
Economia borghese e alienazione

Esamina la critica di Marx all'econo...

Score: -0.0634
Document: Marx e il materialismo storico
Il sogno di Marx
La dittatura del proletariato
Spiega la fase transit...

Score: -0.0635
Document: Marx e il materialismo storico
Il Manifesto del partito comunista
Il socialismo scientifico
Spiega i...

Score: -0.0695
Document: Marx e il materialismo storico
IL SOCIALISMO SCIENTIFICO
Il mondo di Marx e Engels
Lo scenario della...

Score: -0.0772
Document: Marx e il materialismo storico
Il Manifesto del partito comunista
Ri

In [ ]:
results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results]]

In [136]:
SYSTEM_MESSAGE = """Sei un esperto di sistemi educativi. Il tuo compito è determinare il livello di istruzione necessario (scala 0-12) per una lezione basandoti su capitoli di riferimento recuperati.

REGOLE DI RAGIONAMENTO:
1. Logica del Vincolo (Max dei Minimi): Se una lezione tratta più argomenti, il livello MINIMO della lezione deve coincidere con il livello più ALTO tra i minimi dei singoli argomenti. (Esempio: Argomento A liv. 10 + Argomento B liv. 12 = Livello Minimo Lezione 12).
2. Filtro Pertinenza: Usa i dati dei capitoli solo se sono realmente coerenti con l'argomento della lezione. Se i capitoli recuperati sono fuori tema, ignora i loro numeri e stima il livello basandoti sulla complessità concettuale della descrizione.
3. Coerenza del Range: Se la lezione è focalizzata su un singolo tema specifico, il livello minimo e massimo dovrebbero coincidere.
4. Livelli target: 0-4 (Elementari), 5-7 (Medie), 8-12 (Superiori).
5. Gerarchia della Pertinenza: > * Considera "Pertinenti" solo i capitoli che trattano l'argomento centrale della descrizione (es. se la lezione è su Aristotele, i capitoli su Aristotele sono l'unico riferimento valido).

. I capitoli che trattano critiche successive o raccordi storici (es. Bacone che critica Aristotele, o Kant che cita Platone) devono essere considerati "Secondari" e i loro livelli non devono influenzare il calcolo del vincolo, a meno che quegli autori non siano esplicitamente menzionati nella descrizione della lezione.

REGOLE DI ESCLUSIONE CATEGORICA:
- Argomento Centrale: Il calcolo del livello deve basarsi solo su capitoli che condividono lo stesso ambito applicativo della descrizione. 
- Divieto di Approfondimento Forzato: Non alzare il livello minimo usando capitoli "difficili" solo perché trattano temi simili in modo più approfondito. Se la lezione parla di Costituzione, usa i livelli della Costituzione.
"""

In [137]:
PROMPT_TEMPLATE = """### INPUT
Descrizione Lezione: "{{descrizione_lezione}}"
Capitoli Recuperati: 
{{risultati_retriever}}

### ISTRUZIONI
1. Analizza la descrizione e identifica i temi principali.
2. Per ogni tema, verifica se esiste un capitolo recuperato pertinente.
3. Applica la "Logica del Vincolo": identifica il livello minimo più alto tra tutti i temi necessari per comprendere la lezione.
4. Se i capitoli non sono pertinenti, indica "stima_autonoma" e calcola il livello in base alla difficoltà dei concetti espressi.
### FORMATO OUTPUT (JSON)
{
  "livello_min_consigliato": int >= 0,
  "livello_max_consigliato": int <= 12,
  "motivazione": "Spiega quale tema ha determinato il livello minimo finale",
  "fonte_livello": "capitoli_recuperati" O "stima_autonoma"
}"""

In [2]:
df_true = pd.read_excel('../data/V2_elenco-contenuti-editori_AGGIORNATI_da-backend.xlsx')

NameError: name 'pd' is not defined

In [1]:
df_filo = df_true.loc[df_true.Discipline.str.lower().str.contains('cittadinanza')]

NameError: name 'df_true' is not defined

In [141]:
print(len(df_filo))
df_filo.head()

3305


,esId,Link,Editore,Titolo,Descrizione,Tipo,Discipline,Ordine di scuola min,Ordine di scuola max,LIV MIN = LIV MAX,...,Visualizzazioni,Vis. Docenti,Vis. Studenti,Utilizzi,Segnalibri,Like,type,id,data,_score
2,mbrv9oABmJEGg9qZpp1W,https://www.youtube.com/watch?v=mwz6DT4yncE,ABB Italia,"Decarbonizzazione, transizione energetica ed e...","Enrico Ragaini, Senior Principal Engineer, e P...",video,"Economia,Cittadinanza e Costituzione",11.0,12.0,0.0,...,0.0,0.0,0.0,4.0,6.0,1.0,NaN,NaN,NaN,NaN
3,b8IW14ABeWek0aUchcGn,abb/mini-impreseperlambienteelosviluppososteni...,ABB Italia,BUSINESS KIT | Mini-imprese per le aziende e l...,Presentazione in pdf che illustra il concetto ...,"approfondimenti,tools","Economia,Cittadinanza e Costituzione",10.0,12.0,0.0,...,46.0,20.0,3.0,23.0,6.0,0.0,NaN,NaN,NaN,NaN
4,OqiK24ABmJEGg9qZA0id,abb/podcast_marketing_manager_-_powered_by_abb...,ABB Italia,BUSINESS KIT | Marketing manager,In questo podcast by ABB approfondiremo il ruo...,audio_podcast,"Cittadinanza e Costituzione,Economia",10.0,11.0,0.0,...,4.0,1.0,0.0,96.0,9.0,0.0,NaN,NaN,NaN,NaN
5,jqiL24ABmJEGg9qZrVPo,abb/podcast_sustainability_manager_-_powered_b...,ABB Italia,BUSINESS KIT | Sustainability manager,In questo podcast by ABB approfondiremo il ruo...,audio_podcast,"Cittadinanza e Costituzione,Economia",11.0,12.0,0.0,...,4.0,1.0,0.0,12.0,5.0,0.0,NaN,NaN,NaN,NaN
6,i6iK24ABmJEGg9qZ9FCI,abb/podcast_product_manager_-_powered_by_abb (...,ABB Italia,BUSINESS KIT | Product manager,In questo podcast by ABB approfondiremo il ruo...,audio_podcast,"Cittadinanza e Costituzione,Economia",11.0,11.0,1.0,...,1.0,0.0,1.0,5.0,5.0,1.0,NaN,NaN,NaN,NaN


In [142]:
df_filo.Descrizione.iloc[1]

'Presentazione in pdf che illustra il concetto di sostenibilità e come realizzare prodotti sostenibili.'

In [102]:
import os

In [103]:
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

In [104]:
from google import genai
from google.genai import types
import os
import json
from typing import List

In [105]:
GEMINI_MODEL = 'gemini-2.5-flash'
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

In [106]:
from pydantic import BaseModel, Field
from typing import Literal
class LivelloResult(BaseModel):
    livello_min_consigliato: int 
    livello_max_consigliato: int
    motivazione: str = Field(description = "Breve spiegazione del perché questi livelli sono stati scelti in base ai capitoli trovati")
    fonte_livello: Literal["capitoli_recuperati","stima_autonoma_su_descrizione"]
    pertinenza_media_capitoli: Literal["Alta","Media","Bassa"]

In [145]:
def assign_min_max(query):
    dense_vector = next(dense_embedding_model.embed(query)) #next(dense_embedding_model.query_embed(query))
    sparse_vector = next(bm25_embedding_model.embed(query))#next(bm25_embedding_model.query_embed(query))
    late_interaction_vector = next(late_interaction_embedding_model.embed(query)) #next(late_interaction_embedding_model.query_embed(query))
    prefetch = [
            models.Prefetch(
                query=dense_vector,
                using="dense",
                limit=20,
            ),
            models.Prefetch(
                query=models.SparseVector(**sparse_vector.as_object()),
                using="bm25",
                limit=1,
            ),
        ]
    results = client.query_points(
             "rag_capitoli",
            prefetch=prefetch,
            query=late_interaction_vector,
            using="colbert",
            with_payload=True,
            limit=20,
    )
    
    points = results.points
    results = model.rerank(query, [point.payload['descr'] for point in points])
    results = [point.payload for point in points if point.payload['descr'] in [result['document'] for result in results[:10]]]
    
    prompt = PROMPT_TEMPLATE.replace('{{descrizione_lezione}}', query).replace('{{risultati_retriever}}', '\n'.join([f'{index}. Descrizione: {res["descr"]}\nTitolo: {res["titolo"]}\nLivello Minimo: {res["liv_min"]}\nLivello Massimo: {res["liv_max"]}' for index, res in enumerate(results)]))
    parts = [
        types.Part.from_text(text=prompt),
    ]
    content_list = [
        types.Content(
            role='user',
            parts=parts
        )
    ]
    result = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=content_list,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_MESSAGE,
            temperature=0.3,
            response_mime_type='application/json',
            response_schema=LivelloResult)
    )

    return result.parsed


In [146]:
results_min = 0
results_max = 0

for idx in range(0,100):
    print('Descrizione: ' + str(df_filo['Descrizione'].iloc[idx]))
    print('True level min: ' + str(df_filo['Ordine di scuola min'].iloc[idx]))
    print('True level max: ' + str(df_filo['Ordine di scuola max'].iloc[idx]))
    print('-'*100)
    res = assign_min_max(df_filo.Descrizione.iloc[idx])
    print('Predicted level min: ' + str(res.livello_min_consigliato))
    print('Predicted level max: ' + str(res.livello_max_consigliato))
    print('-'*100)

    print('Motivazione: ' + res.motivazione)
    print('fonte_livello: '+ res.fonte_livello)
    print('-'*100)
    results_min += 1 if (df_filo['Ordine di scuola min'].iloc[idx] == res.livello_min_consigliato) else 0
    results_max += 1 if df_filo['Ordine di scuola max'].iloc[idx] == res.livello_max_consigliato else 0




Descrizione: Enrico Ragaini, Senior Principal Engineer, e Paolo Perani, Strategic Business Development Manager, entrambi nel Business Electrification di ABB in Italia, hanno incontrato un gruppo di studenti delle classe superiori ai quali hanno raccontato sfide e opportunità della trasformazione industriale in corso e del suo impatto a livello climatico e sociale.
True level min: 11.0
True level max: 12.0
----------------------------------------------------------------------------------------------------
Predicted level min: 10
Predicted level max: 10
----------------------------------------------------------------------------------------------------
Motivazione: Il livello minimo è determinato dal capitolo 'SVILUPPO ECONOMICO E SOSTENIBILITÀ - Il lavoro e l’ambiente' (Livello Minimo: 10), che è altamente pertinente alla descrizione della lezione riguardante la trasformazione industriale e il suo impatto climatico e sociale. Questo capitolo rappresenta il vincolo più alto tra tutti i c

In [90]:
idx

99